<a target="_blank" href="https://colab.research.google.com/github/Launchpad-Analytics/Workshop-IntroToPython/blob/main/s310/day3/03%20-%20Data%20Cleaning%20Topics.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Day 3 - Data Cleaning Topics

In this notebook, we use the `workshop_participants_east_coast` dataset to practice common data cleaning workflows in Python with pandas and numpy. The examples and exercises below are designed for a 45-minute workshop and assume learners are comfortable with basic Python and spreadsheets.


## Workshop dataset

The dataset is stored in `s310/data/workshop_participants_east_coast.csv` and includes participant registration data with real-world issues such as missing values, sentinel values, inconsistent text formatting, duplicate records, and address/phone variations.

Columns in this dataset:
- `Participant_ID`
- `Name`
- `Age`
- `Phone`
- `Address`
- `City`
- `State`
- `Zip_Code`
- `Workshop_Goal`


In [ ]:
import numpy as np
import pandas as pd
from bs4 import BeautifulSoup

data_csv = 's310/data/workshop_participants_east_coast.csv'
data_xlsx = 's310/data/workshop_participants_east_coast.xlsx'

df = pd.read_csv(data_csv, dtype=str)
df = df.replace({'-999': pd.NA, '': pd.NA})
df['Age'] = pd.to_numeric(df['Age'], errors='coerce')
df['Zip_Code'] = df['Zip_Code'].astype('string')


## Data profiling with pandas

Start by inspecting the shape and the first rows of the dataset, then use profiling methods to understand counts, distributions, and summary statistics.


In [ ]:
print('Shape:', df.shape)
display(df.head())
print('Columns:', df.columns.tolist())
print(''
Non-null counts by column:'')
display(df.count())
print(''
Most frequent states:'')
display(df['State'].value_counts(dropna=False))
print(''
Top workshop goals:'')
display(df['Workshop_Goal'].value_counts(dropna=False).head(10))
print(''
Age summary:'')
display(df['Age'].describe())
print(''
Full dataset summary:'')
display(df.describe(include='all', datetime_is_numeric=True))


## Finding missing values

Use `isna` and `sum` to count missing values. This dataset contains blank fields and placeholder values such as `-999` that should be converted to real nulls.


In [ ]:
missing_by_column = df.isna().sum().sort_values(ascending=False)
display(missing_by_column)
print(''
Rows with missing phone or workshop goals:'')
display(df[df['Phone'].isna() | df['Workshop_Goal'].isna()].head(10))


## Standardize nulls and string types

Convert sentinel values and blank strings into proper missing values, and normalize types for numeric and text columns.


In [ ]:
clean = df.copy()
clean = clean.replace({'-999': pd.NA, 'N/A': pd.NA})
clean['Zip_Code'] = clean['Zip_Code'].str.replace(r'\.0$', '', regex=True)
clean['Zip_Code'] = clean['Zip_Code'].str.zfill(5)
clean['Phone'] = clean['Phone'].astype('string').str.strip()
clean['Workshop_Goal'] = clean['Workshop_Goal'].astype('string').str.strip()
display(clean[['Participant_ID', 'Age', 'Phone', 'Zip_Code', 'Workshop_Goal']].head(10))


## Backfilling, forward filling, and interpolation

These techniques help fill gaps in numeric and text data. Use them carefully, especially when the dataset is not ordered by a meaningful grouping variable.


In [ ]:
clean['Age_ffill'] = clean['Age'].fillna(method='ffill')
clean['Age_bfill'] = clean['Age'].fillna(method='bfill')
clean['Age_interpolate'] = clean['Age'].interpolate()
clean['Workshop_Goal_ffill'] = clean['Workshop_Goal'].fillna(method='ffill')
display(clean[['Age', 'Age_ffill', 'Age_bfill', 'Age_interpolate']].head(15))
print(''
Workshop goal forward fill example:'')
display(clean[['Workshop_Goal', 'Workshop_Goal_ffill']].head(10))


## String formatting and invalid value cleanup

Clean city names, addresses, and workshop goal text so values are consistent. String methods and regular expressions are helpful for standardizing real-world text data.


In [ ]:
clean['City'] = clean['City'].replace({'Philly': 'Philadelphia', 'NewYork': 'New York'})
clean['Address'] = (
    clean['Address']
    .astype('string')
    .str.title()
    .str.replace(r'\bRd\b\.?', 'Road', regex=True)
    .str.replace(r'\bSt\b\.?', 'Street', regex=True)
    .str.replace(r'\bAve\b\.?', 'Avenue', regex=True)
    .str.replace(r'\bBlvd\b\.?', 'Boulevard', regex=True)
)
name_split = clean['Name'].str.split(n=1, expand=True)
clean['First_Name'] = name_split[0]
clean['Last_Name'] = name_split[1]
display(clean[['Name', 'First_Name', 'Last_Name', 'Address', 'City']].head(10))


## Phone number cleaning

Remove non-digit characters from phone numbers, keep only valid 10-digit values, and format them consistently.


In [ ]:
clean['Phone_digits'] = clean['Phone'].astype('string').str.replace(r'\D+', '', regex=True)
clean['Phone_digits'] = clean['Phone_digits'].replace('', pd.NA)
clean['Phone_clean'] = clean['Phone_digits'].where(clean['Phone_digits'].str.len() == 10, pd.NA)
clean['Phone_formatted'] = clean['Phone_clean'].apply(
    lambda x: f'({x[:3]}) {x[3:6]}-{x[6:]}' if pd.notna(x) else pd.NA
)
display(clean[['Phone', 'Phone_digits', 'Phone_clean', 'Phone_formatted']].head(15))


## Identity resolution and duplicate detection

Use duplicate detection on cleaned keys to find repeated registrations and prepare the data for deduplication.


In [ ]:
duplicate_rows = clean[clean.duplicated(subset=['Name', 'Phone_clean', 'Zip_Code'], keep=False)]
print('Number of rows that match by Name + Phone + Zip_Code (keep=False):', len(duplicate_rows))
display(duplicate_rows[['Participant_ID', 'Name', 'Phone_clean', 'Zip_Code', 'Workshop_Goal']].head(20))
deduped = clean.drop_duplicates(subset=['Name', 'Phone_clean', 'Zip_Code'])
print('Shape after dropping duplicates:', deduped.shape)


## Address cleaning for geocoding

Combine cleaned address fields into a single string that can be used for geocoding or mapping.


In [ ]:
final = clean.copy()
final['Full_Address'] = (
    final['Address'].fillna('') + ', ' +
    final['City'].fillna('') + ', ' +
    final['State'].fillna('') + ' ' +
    final['Zip_Code'].fillna('')
)
final['Full_Address'] = (
    final['Full_Address']
    .str.replace(r',\s*,', ',', regex=True)
    .str.replace(r',\s*$', '', regex=True)
    .str.strip()
)
display(final[['Address', 'City', 'State', 'Zip_Code', 'Full_Address']].head(10))


## Importing from Excel and TSV using the same dataset

The same dataset is available in Excel format in `s310/data/workshop_participants_east_coast.xlsx`. We can also convert the cleaned dataset into TSV text and read it back as a demonstration of text-file import.


In [ ]:
print('Excel sheet names:')
excel = pd.ExcelFile(data_xlsx)
print(excel.sheet_names)
excel_df = pd.read_excel(data_xlsx, sheet_name=0, dtype=str)
display(excel_df.head(3))

import io
tsv_text = final.head(5).to_csv(sep='	', index=False)
print('TSV sample:')
print(tsv_text)
tsv_df = pd.read_csv(io.StringIO(tsv_text), sep='	', dtype=str)
display(tsv_df.head(3))


## Simple HTML table scraping example

This is a local example of the same dataset being converted into HTML and then parsed back with BeautifulSoup, which is a common pattern in web scraping.


In [ ]:
html_table = df.head(5).to_html(index=False)
soup = BeautifulSoup(html_table, 'html.parser')
table = soup.find('table')
rows = []
for tr in table.find_all('tr'):
    cells = [td.get_text(strip=True) for td in tr.find_all(['th', 'td'])]
    if cells:
        rows.append(cells)
scraped_df = pd.DataFrame(rows[1:], columns=rows[0])
display(scraped_df.head())


## PDF extraction note

This dataset is available as CSV and Excel in this workshop. If you need to extract the same data from a PDF table, you would typically use a library such as `tabula-py` or `camelot` after installing it. Example code:

```python
import tabula
pdf_path = 'workshop_participants_east_coast.pdf'
pdf_tables = tabula.read_pdf(pdf_path, pages='1', multiple_tables=False)
df_from_pdf = pdf_tables[0]
```


## Workshop exercises

Try the following tasks with this dataset:
1. Count how many rows are missing `Phone` and how many are missing `Workshop_Goal`.
2. Standardize the `City` values for `NewYork` and `Philly` using a mapping.
3. Split `Name` into `First_Name` and `Last_Name`, then count the number of unique first names.
4. Create a cleaned `Full_Address` column and inspect the first 10 rows.
5. Identify duplicate registrations using cleaned phone and zip code values.
6. Save the cleaned dataset to a new CSV file if you want to reuse it in later exercises.
